**Imports , NLKT Setup and Query Tokenizer (From Part 1)**


In [7]:
import pandas as pd
import numpy as np
import json
import math
from collections import defaultdict, Counter
import re, unicodedata
import os
import sys 

# --- NLTK Components for Tokenization ---
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
# Asume que NLTK ya está instalado y descargado (como en tu Notebook de Parte 1)

_STEM = PorterStemmer()
_STOP = set(stopwords.words("english"))
_PUNCT = re.compile(r"[^\w\s]+", re.UNICODE)

def _norm(s: str) -> str:
    """Normalization utility from Part 1 (used for categorical/numeric fields)."""
    if not isinstance(s, str): return ""
    s = unicodedata.normalize("NFKC", s).lower()
    s = re.sub(r"[^\w\s]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def build_terms(text: str) -> list[str]:
    """Applies Part 1 preprocessing to the query (tokenize, stem, filter)."""
    if not isinstance(text, str): return []
    s = unicodedata.normalize("NFKC", text.lower())
    s = _PUNCT.sub(" ", s)
    toks = [t for t in s.split() if t not in _STOP]
    toks = [_STEM.stem(t) for t in toks]
    # Retiene caracteres individuales no numéricos (fix para H&M)
    return [t for t in toks if not t.isdigit()]

def _as_tokens(x):
    if isinstance(x, list):
        return x
    if isinstance(x, np.ndarray):
        return x.tolist()
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return []
    if isinstance(x, str):
        # try JSON list first, else whitespace split
        try:
            v = json.loads(x)
            return v if isinstance(v, list) else [w for w in v.split() if w]
        except Exception:
            return [w for w in x.split() if w]
    return []

**Inverted Index Building**

In [8]:
class InvertedIndex:
    """Stores DF, Posting List (pid, tf), and Document Length (L_d)."""
    def __init__(self):
        # Index: { term: { 'df': int, 'postings': { pid: tf } } }
        self.index = defaultdict(lambda: {'df': 0, 'postings': {}})
        # L_d: { pid: L_d } (Euclidean norm for VSM/Cosine Similarity)
        self.doc_lengths = {}
        self.num_docs = 0

    def add_document(self, doc_id: str, tokens: list[str]):
        """Calculates TF and L_d, and adds terms to postings."""
        tf_counts = Counter(tokens)
        
        # L_d (Euclidean norm of the raw term vector)
        L_d = math.sqrt(sum(tf_counts[term]**2 for term in tf_counts))
        self.doc_lengths[doc_id] = L_d
        
        # Update postings list and document frequency (df)
        for term, tf in tf_counts.items():
            if doc_id not in self.index[term]['postings']:
                self.index[term]['df'] += 1
            self.index[term]['postings'][doc_id] = tf
        
        self.num_docs += 1

    def build_from_dataframe(self, df: pd.DataFrame):
        """Builds index from Part 1 processed data."""
        self.num_docs = len(df)
        
        for _, row in df.iterrows():
            pid = row['pid']
            title = _as_tokens(row['title_tokens'])
            desc  = _as_tokens(row['desc_tokens'])
            details = _as_tokens(row.get('details_tokens', []))
            tokens = title + desc + details
            self.add_document(pid, tokens)
        
        print(f"Index built with {self.num_docs} documents and {len(self.index)} terms.")
    
    def get_term_stats(self, term):
        """Retrieves statistics for a given term."""
        return self.index.get(term, {'df': 0, 'postings': {}})

**TF-IDF Ranking and Retrieval**

In [35]:
def calculate_tfidf_weight(tf, df, N):
    """Calculates the W_t,d or W_t,q TF-IDF weight."""
    # TF Component: 1 + log(tf) (Log-frequency weighting)
    tf_comp = 1 + math.log10(tf) if tf > 0 else 0
    # IDF Component: log(N/df)
    idf_comp = math.log10(N / df) if df > 0 else 0
    return tf_comp * idf_comp

def ranked_search(query: str, index: InvertedIndex) -> list[tuple[str, float]]:
    N = index.num_docs
    terms = build_terms(query)
    if not terms:
        return []

    # AND retrieval
    doc_sets = []
    for t in terms:
        postings = index.get_term_stats(t)['postings']
        if not postings:
            return []  # any missing term -> empty
        doc_sets.append(set(postings.keys()))
    retrieved = list(set.intersection(*doc_sets))

    # Cosine with weighted denominator (over query terms)
    q_tf = Counter(terms)
    scores = {}
    for pid in retrieved:
        num = 0.0
        den = 0.0
        for t in terms:
            stats = index.get_term_stats(t)
            df = stats['df']
            w_d = calculate_tfidf_weight(stats['postings'].get(pid, 0), df, N)
            w_q = calculate_tfidf_weight(q_tf[t], df, N)
            num += w_q * w_d
            den += w_d * w_d
        scores[pid] = (num / math.sqrt(den)) if den > 0 else 0.0

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


**Evaluation Metrics**

In [36]:
def get_relevance_labels(query_id, retrieved_pids, df_labels):
    """Returns ranked binary relevance scores (1/0) and R_total."""
    relevant_labels_df = df_labels[df_labels['query_id'] == query_id]
    relevant_pids = set(relevant_labels_df[relevant_labels_df['relevance'] == 1]['pid'])
    R_total = len(relevant_pids)
    relevance_scores = [1 if pid in relevant_pids else 0 for pid in retrieved_pids]
    return relevance_scores, R_total

# --- Standard Cutoff Metrics ---
def precision_at_k(rel_scores, k):
    """P@K (Required i)"""
    if k == 0 or not rel_scores: return 0.0
    k = min(k, len(rel_scores))
    return sum(rel_scores[:k]) / k

def recall_at_k(rel_scores, R_total, k):
    """R@K (Required ii)"""
    if R_total == 0: return 0.0
    k = min(k, len(rel_scores))
    return sum(rel_scores[:k]) / R_total

def f1_score_at_k(P_at_k, R_at_k):
    """F1-Score@K (Required iv)"""
    if P_at_k + R_at_k == 0: return 0.0
    return 2 * P_at_k * R_at_k / (P_at_k + R_at_k)

# --- Ranking Metrics ---
def average_precision_at_k(rel_scores, k):
    """AP@K (Required iii)"""
    if not rel_scores: return 0.0
    k = min(k, len(rel_scores))
    sum_of_precisions = 0.0
    num_relevant = 0
    for i in range(k):
        if rel_scores[i] == 1:
            num_relevant += 1
            sum_of_precisions += num_relevant / (i + 1)
    return sum_of_precisions / num_relevant if num_relevant > 0 else 0.0

def mean_average_precision(aps_list):
    """MAP (Required v)"""
    return sum(aps_list) / len(aps_list) if aps_list else 0.0

def mean_reciprocal_rank(rel_scores: list):
    """MRR (Required vi)"""
    for i, rel in enumerate(rel_scores):
        if rel == 1:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(rel_scores, k):
    """NDCG@K (Required vii)"""
    k = min(k, len(rel_scores))
    
    # DCG (Actual Ranking)
    dcg = sum(rel_scores[i] / math.log2(i + 2) for i in range(k))
    
    # IDCG (Ideal Ranking)
    ideal_scores = sorted(rel_scores, reverse=True)
    idcg = sum(ideal_scores[i] / math.log2(i + 2) for i in range(k))
        
    return dcg / idcg if idcg > 0.0 else 0.0

**Execution - Build Index and Define Queries**

In [37]:
from pathlib import Path
import os, json, math, pandas as pd

# 1) Define base directory
BASE = Path(r"C:\Users\Ramona\.vscode\IR Project\IRWA_Final_Project")

# 2) Define files directly
PROCESSED_DATA_FILE = BASE / r"data\processed\products_clean.parquet"
LABELS_FILE         = BASE / r"data\raw\validation_labels.csv"
INDEX_FILE          = BASE / r"data\index\inverted_index.json"

# 3) Sanity checks
print("processed:", PROCESSED_DATA_FILE.exists(), PROCESSED_DATA_FILE)
print("labels   :", LABELS_FILE.exists(), LABELS_FILE)
os.makedirs(INDEX_FILE.parent, exist_ok=True)

# 4) Load data 
try:
    df_clean = pd.read_parquet(PROCESSED_DATA_FILE)
except Exception as e:
    print("pandas read_parquet failed:", e)
    import pyarrow.dataset as ds
    df_clean = ds.dataset(PROCESSED_DATA_FILE, format="parquet").to_table().to_pandas()

# 5) Minimal schema check
need_cols = {"pid","title_tokens","desc_tokens"}
missing = need_cols - set(df_clean.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# 6) Build index
INDEX = InvertedIndex()
INDEX.build_from_dataframe(df_clean)
with open(INDEX_FILE, "w") as f:
    json.dump({"index": dict(INDEX.index), "doc_lengths": dict(INDEX.doc_lengths)}, f)
print(f"Index saved: {INDEX_FILE} | docs={INDEX.num_docs}, terms={len(INDEX.index)}")

processed: True C:\Users\Ramona\.vscode\IR Project\IRWA_Final_Project\data\processed\products_clean.parquet
labels   : True C:\Users\Ramona\.vscode\IR Project\IRWA_Final_Project\data\raw\validation_labels.csv
Index built with 56160 documents and 18132 terms.
Index saved: C:\Users\Ramona\.vscode\IR Project\IRWA_Final_Project\data\index\inverted_index.json | docs=56160, terms=18132


**Load Labels**

In [38]:
import pandas as pd

# Load schema
df_labels = pd.read_csv(LABELS_FILE, dtype={'query_id': int, 'pid': str})
if 'labels' in df_labels.columns and 'relevance' not in df_labels.columns:
    df_labels = df_labels.rename(columns={'labels': 'relevance'})
if 'title' in df_labels.columns:
    df_labels = df_labels.drop(columns=['title'])

# Enforce types and values
df_labels = df_labels.astype({'query_id': int, 'pid': str, 'relevance': int})

# Sanity check
need = {"query_id", "pid", "relevance"}
missing = need - set(df_labels.columns)
if missing:
    raise ValueError(f"validation_labels.csv missing columns: {missing}")
print("labelled query_ids:", sorted(df_labels["query_id"].unique().tolist()))
# show first 3 rows
print(df_labels.head(3))


labelled query_ids: [1, 2]
                pid  query_id  relevance
0  SWSFFVKBCQG5FHPF         1          1
1  SWSFJY5ZFHQ7HXKW         1          0
2  SWSFUY89NHMZHZPX         1          1


**Show most frequent terms for building new queries**

In [39]:
term_freqs = {term: len(stats['postings']) for term, stats in INDEX.index.items()}
top_terms = sorted(term_freqs.items(), key=lambda x: x[1], reverse=True)[:30]
print(top_terms)

[('color', 26865), ('fabric', 26450), ('style', 26042), ('pattern', 25783), ('code', 25764), ('care', 22642), ('pack', 21842), ('wear', 20791), ('wash', 20528), ('cotton', 19933), ('suitabl', 19309), ('western', 19118), ('sleev', 18901), ('fit', 18837), ('type', 18189), ('ideal', 17198), ('shirt', 16783), ('name', 15785), ('machin', 15760), ('regular', 15400), ('neck', 15190), ('revers', 14819), ('packag', 14276), ('sale', 14266), ('women', 13817), ('brand', 13689), ('blend', 13593), ('men', 13501), ('size', 13253), ('gener', 12303)]


**Define Queries**

In [40]:
ALL_QUERIES = [
    {'query_id': 1, 'query': "women full sleeve sweatshirt cotton"},
    {'query_id': 2, 'query': "men slim jeans blue"},
    {'query_id': 3, 'query': "long sleeve denim jacket blue"},
    {'query_id': 4, 'query': "cotton shirt man regular fit"},
    {'query_id': 5, 'query': "women western wear cotton"},
    {'query_id': 6, 'query': "machine wash suitabl woman"},
    {'query_id': 7, 'query': "brand blend fabric shirt"},
]
K = 10


**Ranking Results for Queries 1 and 2**

In [41]:
# top 10 results for each query
def show_topk(q, k=10):
    ranked = ranked_search(q, INDEX)[:k]
    df = df_clean.set_index('pid')
    rows = []
    for pid, s in ranked:
        title = df.loc[pid, 'title_tokens'] if pid in df.index else ''
        rows.append({'rank': len(rows)+1, 'pid': pid, 'title': title, 'score': round(s,3)})
    return pd.DataFrame(rows)

for q in [
    "women full sleeve sweatshirt cotton",
    "men slim jeans blue",
]:
    print(q); display(show_topk(q))

# worked out example token
term = "sweatshirt"; stats = INDEX.get_term_stats(term)
df_t = stats['df']; N = INDEX.num_docs
pid = ranked_search("women full sleeve sweatshirt cotton", INDEX)[0][0]
tf_d = stats['postings'].get(pid, 0)
tf_q = Counter(build_terms("women full sleeve sweatshirt cotton"))[term]
idf = math.log10(N/df_t); w_d = (1+math.log10(tf_d))*idf; w_q = (1+math.log10(tf_q))*idf
contrib = w_d*w_q
print({"term":term,"df":df_t,"idf":idf,"tf_d":tf_d,"w_d":w_d,"tf_q":tf_q,"w_q":w_q,"dot_contrib":contrib})


women full sleeve sweatshirt cotton


,rank,pid,title,score
0,1,SWSFXNFSHFZKXEZE,"[full, sleev, graphic, print, women, sweatshirt]",2.036
1,2,SWSFUY89DXWXDHUG,"[full, sleev, print, women, sweatshirt]",2.035
2,3,SWSFXNFSEY9HJZFA,"[full, sleev, print, women, sweatshirt]",2.034
3,4,SWSFXNFSSJVJA9HU,"[full, sleev, graphic, print, women, sweatshirt]",2.034
4,5,SWSFXNFSTFHBSSKZ,"[full, sleev, graphic, print, women, sweatshirt]",2.034
5,6,SWSFZDN8ZQZZH9FJ,"[full, sleev, graphic, print, women, sweatshirt]",2.034
6,7,SWSFXNFS7CY3GSXE,"[full, sleev, graphic, print, women, sweatshirt]",2.034
7,8,SWSFXNFSBWAC3YZH,"[full, sleev, graphic, print, women, sweatshirt]",2.034
8,9,SWSFWEFFG8A5DGZA,"[full, sleev, graphic, print, women, sweatshirt]",2.034
9,10,SWSFWG9BXWWDZVCA,"[full, sleev, color, block, women, sweatshirt]",2.033


men slim jeans blue


,rank,pid,title,score
0,1,JEAECSRFKPAGZ9WS,"[slim, men, dark, blue, jean]",2.006
1,2,JEAECSRFHAUAWZ9R,"[slim, men, dark, blue, jean]",2.006
2,3,JEAF2QF6HNZHDXVX,"[slim, men, dark, blue, jean]",2.005
3,4,JEAEHGRJSGGYEYYX,"[slim, men, light, blue, jean]",2.004
4,5,JEAERYGSGHGM9JBA,"[slim, men, blue, jean]",2.003
5,6,JEAERZEPZXK2MADH,"[slim, men, dark, blue, jean]",2.003
6,7,JEAFGJ9Y9QKCYA6J,"[slim, men, blue, jean]",2.003
7,8,JEAEKMCHWJJVUPQE,"[slim, men, blue, jean]",2.003
8,9,JEAERYJ25KBEYNTG,"[slim, men, blue, jean]",2.003
9,10,JEAFSKYHDYZK5SHZ,"[slim, men, blue, jean]",2.002


{'term': 'sweatshirt', 'df': 1418, 'idf': 1.597750868274701, 'tf_d': 3, 'w_d': 2.3600717672753566, 'tf_q': 1, 'w_q': 1.597750868274701, 'dot_contrib': 3.770806715354809}


**Evaluation of Ranking Results for Queries 1 and 2**

In [48]:
df_labels['pid'] = df_labels['pid'].str.upper()
df_clean['pid'] = df_clean['pid'].str.upper()

K = 10

# Only queries 1 and 2
EVAL_QUERIES = [q for q in ALL_QUERIES if q['query_id'] in (1, 2)]

print("\n--- 2. RUNNING RANKED SEARCH AND EVALUATION (Q1–Q2) ---")
RESULTS, AP_SCORES = [], []

for q in EVAL_QUERIES:
    qid, qtext = q['query_id'], q['query']
    ranked = ranked_search(qtext, INDEX)
    retrieved_pids = [pid for pid, _ in ranked]

    rel_scores, R_total = get_relevance_labels(qid, retrieved_pids, df_labels)

    Pk   = precision_at_k(rel_scores, K)
    Rk   = recall_at_k(rel_scores, R_total, K)
    F1k  = f1_score_at_k(Pk, Rk)
    APk  = average_precision_at_k(rel_scores, K)
    MRR  = mean_reciprocal_rank(rel_scores)
    NDCG = ndcg_at_k(rel_scores, K)

    AP_SCORES.append(APk)
    RESULTS.append({
        'Query ID': qid, 'Query Text': qtext, 'R_Total': R_total,
        'Retrieved': len(retrieved_pids), 'P@10': round(Pk,3),
        'R@10': round(Rk,3), 'F1@10': round(F1k,3),
        'AP@10': round(APk,3), 'MRR': round(MRR,3), 'NDCG@10': round(NDCG,3)
    })

df_results = pd.DataFrame(RESULTS)
print(df_results.to_string(index=False))
print(f"\nMAP over {len(EVAL_QUERIES)} queries: {round(mean_average_precision(AP_SCORES), 3)}")




--- 2. RUNNING RANKED SEARCH AND EVALUATION (Q1–Q2) ---
 Query ID                          Query Text  R_Total  Retrieved  P@10  R@10  F1@10  AP@10   MRR  NDCG@10
        1 women full sleeve sweatshirt cotton       13        500   0.0   0.0    0.0    0.0 0.043      0.0
        2                 men slim jeans blue       10        222   0.0   0.0    0.0    0.0 0.022      0.0

MAP over 2 queries: 0.0


**Ground Truth for New Queries**

In [8]:

# The assignment requires you to manually define the ground truth for Q3-Q7 
# and update your validation_labels.csv file. 

# ACTION REQUIRED:
# 1. Inspect the PIDs retrieved by the search engine for Queries 3 through 7.
# 2. For those PIDs, manually judge relevance (1 or 0).
# 3. Add these new relevance judgments to your data/raw/validation_labels.csv file.
# 4. Include a detailed table of your manual judgments in your final PDF report.

# Example:
# Query 3: 'long sleeve denim jacket blue'
# - PID_12345: Relevant (1) because it is a denim jacket.
# - PID_67890: Not Relevant (0) because it is a denim dress.


**Inspect top results**

In [25]:
for q in ALL_QUERIES[2:]:
    print(q['query'])
    ranked = ranked_search(q['query'], INDEX)
    print([pid for pid,_ in ranked[:10]])
    print()


long sleeve denim jacket blue
['JCKFS4NFYXFRDWDH', 'JCKFPEVFF5DWZB4H', 'JCKFT9TE2DB6TJNQ', 'JCKFKKFZ7QB2KUQV', 'JCKFKKFZMDEHEZQS', 'SHTFRVPAV33BCXEZ', 'JCKFEED3BHMKFAYJ']

cotton shirt man regular fit
['TSHFNV35W6XMETKT', 'SHTFT26KX5DEYK4H', 'SHTFT26K94NBMH9Q', 'SHTFT26KAPZNN3EH', 'SHTFT26GMQAS7PYP', 'SHTFT26G9JRR3RZF', 'SHTFT26JQX46F3XR', 'SHTFT26GVKGRAGAF', 'SHTFT26JT9GYDGYQ', 'SHTFT26GHCTVDQFJ']

women western wear cotton
['JEAFF3WMFAVHRGWD', 'JEAFF3WMUVBPCH2C', 'JEAFVM7CQV978YUA', 'JEAFEN3WGGV3G4NK', 'JEAFFYG8MGJA7UEZ', 'JEAFFYG8HXHCSDWG', 'JEAFFYG8FP3JZUZ6', 'JEAFF4YGZZFGFCVJ', 'JEAFFYG8GQDQRZHR', 'JEAFF4YGVUA9MN9W']

machine wash suitabl woman
['TSHFNZMHMCGHYKRX', 'TSHFNT4PFGSWUCH6', 'TSHFNV35EAWAU2HJ', 'TSHFNT5FEZH9GFST', 'TSHFNSFPGGZZVKTP', 'TSHFNV489PZZ8TCK', 'TSHFNV48NWATWFEW', 'TSHFXBBWTSNYV9HE', 'TSHFXCXZSBH5UFJT', 'TSHFW4S4JVEZWDAM']

brand blend fabric shirt
['TSHF2DVYEJAB8ZSU', 'TSHF2DVYJXXPBHSG', 'TSHFF4WR8GZAZYXQ', 'TSHF2DVYRV2RF5ZV', 'TSHFF4WRWHNZCHUV', 'TSHF2DVY4D9Y3